# 05 · Fine-tuning para classificação (spam) — *Elmo*

**Entra:** GPT-2 124M pré-treinado (04). **Sai:** um classificador spam / não-spam.

Ideia central: trocamos a cabeça de saída (50.257 tokens → **2 classes**) e treinamos pouca coisa por cima do que o GPT já sabe.

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import tiktoken
import torch
from torch.utils.data import DataLoader

from aula import DATA, CHECKPOINTS, device, load_gpt2, n_params
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text
from llms_from_scratch.ch06 import (download_and_unzip_spam_data, create_balanced_dataset, random_split,
                                    SpamDataset, calc_accuracy_loader, train_classifier_simple, classify_review)

# 🔧 True = usa o classificador salvo (se existir) em vez de treinar ao vivo
CARREGAR_CHECKPOINT = True
N_EPOCAS = 5
bpe = tiktoken.get_encoding("gpt2")

## 1. Dados: SMS Spam Collection (UCI)

In [2]:
tsv = Path(DATA) / "SMSSpamCollection.tsv"
download_and_unzip_spam_data("https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip",
                             Path(DATA) / "sms_spam_collection.zip", DATA, tsv)
df = pd.read_csv(tsv, sep="\t", header=None, names=["Label", "Text"])
print(df["Label"].value_counts(), "\n")

df = create_balanced_dataset(df)  # mesmo número de spam e ham
df["Label"] = df["Label"].map({"ham": 0, "spam": 1})
treino, val, teste = random_split(df, 0.7, 0.1)
for nome, parte in [("train", treino), ("validation", val), ("test", teste)]:
    parte.to_csv(Path(DATA) / f"{nome}.csv", index=None)
df.sample(5, random_state=1)

/Users/victor/Documents/masters/gen AI/LLMs-from-scratch/30-09/data/SMSSpamCollection.tsv already exists. Skipping download and extraction.
Label
ham     4825
spam     747
Name: count, dtype: int64 



,Label,Text
733,0,Lol you won't feel bad when I use her money to...
952,0,Shb b ok lor... Thanx...
3443,1,Save money on wedding lingerie at www.bridal.p...
3415,0,No pic. Please re-send.
1466,1,YOU 07801543489 are guaranteed the latests Nok...


In [3]:
train_ds = SpamDataset(Path(DATA) / "train.csv", bpe)  # padding até a maior mensagem
val_ds = SpamDataset(Path(DATA) / "validation.csv", bpe, max_length=train_ds.max_length)
test_ds = SpamDataset(Path(DATA) / "test.csv", bpe, max_length=train_ds.max_length)
torch.manual_seed(123)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=8)
test_loader = DataLoader(test_ds, batch_size=8)
print(len(train_ds), "treino |", len(val_ds), "validação |", len(test_ds), "teste | max_length =", train_ds.max_length)

1045 treino | 149 validação | 300 teste | max_length = 120


## 2. Dá para classificar só com o GPT-2 pré-treinado, via prompt?

In [4]:
gpt2, cfg = load_gpt2("124M")
gpt2.to(device)
prompt = ("Is the following text 'spam'? Answer with 'yes' or 'no': "
          "'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'")
out = generate(gpt2, text_to_token_ids(prompt, bpe).to(device), max_new_tokens=20, context_size=cfg["context_length"])
print(token_ids_to_text(out, bpe)[len(prompt):])



The following text 'spam'? Answer with 'yes' or 'no': 'You


Não: o modelo base só **continua texto**. Ele nunca foi ensinado a seguir instruções (isso é o notebook 06).

## 3. Adaptando a arquitetura
1. congela tudo  2. troca `out_head` por `Linear(768, 2)`  3. libera o último bloco e o LayerNorm final.

In [5]:
for p in gpt2.parameters():
    p.requires_grad = False
torch.manual_seed(123)
gpt2.out_head = torch.nn.Linear(cfg["emb_dim"], 2).to(device)
for p in list(gpt2.trf_blocks[-1].parameters()) + list(gpt2.final_norm.parameters()):
    p.requires_grad = True
print(f"treináveis: {n_params(gpt2, treinaveis=True):,} de {n_params(gpt2):,}")

treináveis: 7,090,946 de 124,441,346


In [6]:
# 🔧 Quanto treinar? Contagem de parâmetros treináveis em cada estratégia
blocos = gpt2.trf_blocks
por_bloco = sum(p.numel() for p in blocos[0].parameters())
for nome, n in [("só a cabeça", 768 * 2 + 2), ("cabeça + último bloco", 768 * 2 + 2 + por_bloco),
                ("tudo (full fine-tuning)", n_params(gpt2))]:
    print(f"{nome:25s} {n:>12,}")

só a cabeça                      1,538
cabeça + último bloco        7,089,410
tudo (full fine-tuning)    124,441,346


Usamos só o **último token** da saída: pela máscara causal, ele é o único que "viu" a mensagem inteira.

In [7]:
with torch.no_grad():
    print("saída do modelo:", tuple(gpt2(next(iter(train_loader))[0].to(device)).shape), "→ usamos [:, -1, :]")
torch.manual_seed(123)
print(f"acurácia ANTES do treino — val: {calc_accuracy_loader(val_loader, gpt2, device, num_batches=10):.0%}")

saída do modelo: (8, 120, 2) → usamos [:, -1, :]


acurácia ANTES do treino — val: 45%


## 4. Treino (ou checkpoint)

In [8]:
ckpt = os.path.join(CHECKPOINTS, "classificador_spam.pth")
if CARREGAR_CHECKPOINT and os.path.exists(ckpt):
    gpt2.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True), strict=False)
    print("checkpoint carregado:", ckpt)
else:
    torch.manual_seed(123)
    t0 = time.time()
    optimizer = torch.optim.AdamW(gpt2.parameters(), lr=5e-5, weight_decay=0.1)
    train_classifier_simple(gpt2, train_loader, val_loader, optimizer, device,
                            num_epochs=N_EPOCAS, eval_freq=50, eval_iter=5)
    print(f"{(time.time() - t0) / 60:.1f} min")
    # salva só o que foi treinado (~28 MB); o resto é o GPT-2 original
    torch.save({n: p.detach() for n, p in gpt2.named_parameters() if p.requires_grad}, ckpt + ".part")
    os.replace(ckpt + ".part", ckpt)

Ep 1 (Step 000000): Train loss 2.153, Val loss 2.392


Ep 1 (Step 000050): Train loss 0.617, Val loss 0.637


Ep 1 (Step 000100): Train loss 0.523, Val loss 0.557


Training accuracy: 70.00% | Validation accuracy: 72.50%


Ep 2 (Step 000150): Train loss 0.561, Val loss 0.489


...

Ep 4 (Step 000500): Train loss 0.222, Val loss 0.137


Training accuracy: 100.00% | Validation accuracy: 97.50%


Ep 5 (Step 000550): Train loss 0.207, Val loss 0.143


Ep 5 (Step 000600): Train loss 0.083, Val loss 0.074


Training accuracy: 100.00% | Validation accuracy: 97.50%
2.0 min


In [9]:
for nome, loader in [("treino", train_loader), ("validação", val_loader), ("teste", test_loader)]:
    print(f"acurácia {nome:10s}: {calc_accuracy_loader(loader, gpt2, device):.1%}")

acurácia treino    : 97.2%


acurácia validação : 97.3%


acurácia teste     : 95.7%


## 5. Testando com mensagens da turma ✍️

In [10]:
mensagens = [
    "You are a winner you have been specially selected to receive $1000 cash or a $2000 award.",
    "Hey, just wanted to check if we're still on for dinner tonight? Let me know!",
    "URGENT! Your account has been suspended. Click here to verify your password now.",
    "Can you send me the slides from class before Friday?",
]
for m in mensagens:
    print(f"{classify_review(m, gpt2, bpe, device, max_length=train_ds.max_length):9s} ← {m}")

spam      ← You are a winner you have been specially selected to receive $1000 cash or a $2000 award.
not spam  ← Hey, just wanted to check if we're still on for dinner tonight? Let me know!
spam      ← URGENT! Your account has been suspended. Click here to verify your password now.
not spam  ← Can you send me the slides from class before Friday?


### ✅ Construto
Um GPT-2 com cabeça de 2 classes. O mesmo truque serve para sentimento, tópicos, intenção… basta trocar o nº de saídas.
Mais experimentos (treinar tudo, usar o primeiro token, LoRA): `ch06/02_bonus_additional-experiments`.